# Chest X-Ray: Data Loading, Filtering & EDA
This notebook focuses strictly on the data preprocessing pipeline. We will explore the raw dataset, filter the labels, split the data without patient leakage, and visualize sample X-Rays.

In [ ]:
!pip install pandas matplotlib seaborn
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set plot style
sns.set_theme(style="whitegrid")

## 1. Data Configuration

In [ ]:
# Update these paths if running on Kaggle or Colab
DATA_DIR = './nih-chest-xrays/data'
CSV_PATH = os.path.join(DATA_DIR, 'Data_Entry_2017.csv')
IMAGE_DIR = os.path.join(DATA_DIR, 'images')
TRAIN_VAL_LIST = os.path.join(DATA_DIR, 'train_val_list.txt')
TEST_LIST = os.path.join(DATA_DIR, 'test_list.txt')

TARGET_CONDITIONS = ['Pneumonia', 'Effusion', 'Atelectasis', 'Cardiomegaly', 'Pneumothorax']

## 2. Loading the Raw Labels

In [ ]:
# Load raw dataset
print("Loading Data_Entry_2017.csv...")
try:
    df = pd.read_csv(CSV_PATH)
    print(f"Total raw images in CSV: {len(df)}")
    display(df.head())
except FileNotFoundError:
    print("Dataset not found. Please ensure the CSV_PATH is correct.")
    df = pd.DataFrame() # Empty for offline viewing

## 3. Filtering to Target Conditions

In [ ]:
if not df.empty:
    def get_target_labels(labels_str):
        labels = labels_str.split('|')
        has_target = any(cond in labels for cond in TARGET_CONDITIONS)
        has_no_finding = 'No Finding' in labels
        if has_target or has_no_finding:
            return True
        return False

    filtered_df = df[df['Finding Labels'].apply(get_target_labels)].copy()
    print(f"Images after filtering for target conditions + 'No Finding': {len(filtered_df)}")
    
    # Create binary columns
    for condition in TARGET_CONDITIONS:
        filtered_df[condition] = filtered_df['Finding Labels'].apply(lambda x: 1.0 if condition in x else 0.0)
        
    filtered_df['No Finding'] = filtered_df['Finding Labels'].apply(lambda x: 1.0 if 'No Finding' in x else 0.0)

## 4. Class Distribution Analysis

In [ ]:
if not df.empty:
    class_counts = {}
    for condition in TARGET_CONDITIONS:
        class_counts[condition] = filtered_df[condition].sum()
    class_counts['No Finding'] = filtered_df['No Finding'].sum()

    plt.figure(figsize=(10, 6))
    sns.barplot(x=list(class_counts.keys()), y=list(class_counts.values()), palette="viridis")
    plt.title('Class Distribution After Filtering', fontsize=16)
    plt.ylabel('Number of Images', fontsize=12)
    plt.xticks(rotation=45)
    plt.show()
    
    for k, v in class_counts.items():
        print(f"{k}: {int(v)}")

## 5. Patient-Level Splitting
To prevent data leakage, we must ensure that images from the same patient do not appear in both the training and testing sets. We use the official `train_val_list.txt` and `test_list.txt`.

In [ ]:
if not df.empty and os.path.exists(TRAIN_VAL_LIST) and os.path.exists(TEST_LIST):
    with open(TRAIN_VAL_LIST, 'r') as f:
        train_val_images = [line.strip() for line in f.readlines()]
    with open(TEST_LIST, 'r') as f:
        test_images = [line.strip() for line in f.readlines()]
        
    train_val_df = filtered_df[filtered_df['Image Index'].isin(train_val_images)]
    test_df = filtered_df[filtered_df['Image Index'].isin(test_images)]
    
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_val_df, test_size=0.176, random_state=42)
    
    print(f"Train subset: {len(train_df)} images")
    print(f"Validation subset: {len(val_df)} images")
    print(f"Test subset: {len(test_df)} images")

## 6. Visualizing Sample X-Rays
Let's look at 4 sample images from the dataset and their associated multi-labels.

In [ ]:
if not df.empty and os.path.exists(IMAGE_DIR):
    # Pick 4 random images
    sample_df = filtered_df.sample(4, random_state=42)
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    for i, (idx, row) in enumerate(sample_df.iterrows()):
        img_name = row['Image Index']
        img_path = os.path.join(IMAGE_DIR, img_name)
        
        try:
            image = Image.open(img_path).convert('RGB')
            axes[i].imshow(image)
            
            # Determine active labels
            active = [cond for cond in TARGET_CONDITIONS if row[cond] == 1.0]
            if row['No Finding'] == 1.0:
                active.append('No Finding')
                
            title = "\n".join(active)
            axes[i].set_title(title, fontsize=12)
            axes[i].axis('off')
        except FileNotFoundError:
            axes[i].set_title("Image not found")
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.show()
else:
    print("Images directory not found, cannot visualize samples.")